# Experiment 1: Free Shipping Threshold

**Hypothesis:** Lowering the free-shipping threshold from R$50 to R$35 increases the conversion rate.

| Property | Value |
|---|---|
| Experiment ID | `exp_001` |
| Primary metric | Conversion rate |
| Secondary metrics | AOV, items per order, revenue per user |
| Guardrail metrics | Profit margin, return rate, review score |
| Control | Free shipping at R$50 (`control_50`) |
| Treatment | Free shipping at R$35 (`treatment_35`) |
| MDE | 10% relative lift |
| Planned sample | 2,450 per variant |
| Test type | Chi-square / Z-test for proportions |
| Randomisation unit | `customer_id` |

**Business context:** Free-shipping thresholds directly affect purchase
decisions. A lower bar may boost conversion but increases shipping cost
exposure. The experiment quantifies whether the revenue lift from
additional orders outweighs the extra freight subsidy.

In [ ]:
import sys, warnings, math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.power import NormalIndPower

ROOT = Path.cwd() if "notebooks" not in str(Path.cwd()) else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ecommerce_analytics.analysis import (
    ExperimentAnalyzer,
    ExperimentCatalog,
    load_experiment_data,
)
from ecommerce_analytics.analysis.stats_framework import (
    cohens_h,
    interpret_effect_size,
    _ci_proportion_diff,
    _ci_mean_diff,
    achieved_power_proportions,
    required_sample_size,
)

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)

ALPHA = 0.05
CONFIDENCE = 0.95
MIN_PRACTICAL_LIFT = 10.0  # %
MONTHLY_TRAFFIC = 100_000

---
## Section 1 — Data Extraction

### BigQuery Reference Query

In production this data would come from BigQuery:

```sql
WITH experiment_data AS (
  SELECT
    a.customer_id,
    a.experiment_id,
    a.variant,
    o.order_id,
    o.order_total,
    o.item_count,
    o.freight_value,
    CASE WHEN o.order_id IS NOT NULL THEN 1 ELSE 0 END AS converted
  FROM `marts.experiment_assignments` a
  LEFT JOIN `marts.fct_orders` o
    ON a.customer_id = o.customer_id
  WHERE a.experiment_id = 'exp_001'
)
SELECT
  variant,
  COUNT(DISTINCT customer_id)                           AS users,
  SUM(converted)                                        AS conversions,
  SAFE_DIVIDE(SUM(converted), COUNT(DISTINCT customer_id)) AS conversion_rate,
  AVG(IF(converted = 1, order_total, NULL))             AS avg_order_value,
  SUM(order_total)                                      AS total_revenue,
  AVG(IF(converted = 1, freight_value, NULL))           AS avg_freight
FROM experiment_data
GROUP BY variant;
```

For this portfolio project we load locally via the analysis framework.

In [ ]:
catalog = ExperimentCatalog()
meta = catalog.get("exp_001")

data = load_experiment_data("exp_001")

control_df = data[data["variant"] == "control"].copy()
treatment_df = data[data["variant"] == "treatment"].copy()

print(f"Total rows      : {len(data):,}")
print(f"Control (n)     : {len(control_df):,}")
print(f"Treatment (n)   : {len(treatment_df):,}")
print(f"Balance         : {len(control_df)/len(data):.1%} / {len(treatment_df)/len(data):.1%}")
print(f"Columns         : {list(data.columns)}")
data.head(3)

In [ ]:
# Randomisation balance check — covariate balance between variants
if "customer_state" in data.columns:
    top_states = data["customer_state"].value_counts().head(5).index
    balance = (
        data[data["customer_state"].isin(top_states)]
        .groupby(["variant", "customer_state"])
        .size()
        .unstack(fill_value=0)
    )
    balance_pct = balance.div(balance.sum(axis=1), axis=0)
    print("Top-5 state distribution by variant (should be similar):")
    display(balance_pct.round(3))

---
## Section 2 — Descriptive Statistics

In [ ]:
n_ctrl = len(control_df)
n_treat = len(treatment_df)

conv_ctrl = int(control_df["converted"].sum())
conv_treat = int(treatment_df["converted"].sum())

cr_ctrl = conv_ctrl / n_ctrl
cr_treat = conv_treat / n_treat

abs_lift = cr_treat - cr_ctrl
rel_lift = abs_lift / cr_ctrl * 100

# Converters only — for AOV and secondary metrics
ctrl_conv = control_df[control_df["converted"]]
treat_conv = treatment_df[treatment_df["converted"]]

summary = pd.DataFrame({
    "Metric": [
        "Sample size",
        "Conversions",
        "Conversion rate",
        "AOV (converters)",
        "Median order value",
        "Std order value",
        "Items per order (converters)",
        "Revenue per user (all)",
    ],
    "Control": [
        f"{n_ctrl:,}",
        f"{conv_ctrl:,}",
        f"{cr_ctrl:.2%}",
        f"R${ctrl_conv['order_total'].mean():,.2f}",
        f"R${ctrl_conv['order_total'].median():,.2f}",
        f"R${ctrl_conv['order_total'].std():,.2f}",
        f"{ctrl_conv['item_count'].mean():.2f}",
        f"R${control_df['order_total'].mean():,.2f}",
    ],
    "Treatment": [
        f"{n_treat:,}",
        f"{conv_treat:,}",
        f"{cr_treat:.2%}",
        f"R${treat_conv['order_total'].mean():,.2f}",
        f"R${treat_conv['order_total'].median():,.2f}",
        f"R${treat_conv['order_total'].std():,.2f}",
        f"{treat_conv['item_count'].mean():.2f}",
        f"R${treatment_df['order_total'].mean():,.2f}",
    ],
})
summary.style.hide(axis="index")

In [ ]:
print(f"Absolute lift : {abs_lift:.4f}  ({abs_lift*100:+.2f} pp)")
print(f"Relative lift : {rel_lift:+.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# ── 2a  Conversion rate bar chart ──────────────────────────────────
ax = axes[0]
bars = ax.bar(
    ["Control\n(R$50 threshold)", "Treatment\n(R$35 threshold)"],
    [cr_ctrl, cr_treat],
    color=["#5975a4", "#cc8963"],
    width=0.5, edgecolor="white", linewidth=1.5,
)
for bar, val in zip(bars, [cr_ctrl, cr_treat]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
            f"{val:.2%}", ha="center", fontweight="bold", fontsize=12)
ax.set_ylabel("Conversion Rate")
ax.set_title("Primary Metric: Conversion Rate", fontweight="bold")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylim(0, max(cr_ctrl, cr_treat) * 1.3)

# ── 2b  Revenue distribution (converters) ─────────────────────────
ax = axes[1]
bins = np.linspace(0, ctrl_conv["order_total"].quantile(0.98), 50)
ax.hist(ctrl_conv["order_total"], bins=bins, alpha=0.6,
        label="Control", color="#5975a4", density=True)
ax.hist(treat_conv["order_total"], bins=bins, alpha=0.6,
        label="Treatment", color="#cc8963", density=True)
ax.set_title("Order Value Distribution (converters)", fontweight="bold")
ax.set_xlabel("Order Total (R$)")
ax.set_ylabel("Density")
ax.legend()

# ── 2c  Box plot ───────────────────────────────────────────────────
ax = axes[2]
conv_data = data[data["converted"]].copy()
upper = conv_data["order_total"].quantile(0.95)
conv_data_clipped = conv_data[conv_data["order_total"] <= upper]
sns.boxplot(data=conv_data_clipped, x="variant", y="order_total", ax=ax,
            palette=["#5975a4", "#cc8963"], width=0.4)
ax.set_title("Order Value Box Plot (< 95th pctl)", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Order Total (R$)")
ax.set_xticklabels(["Control", "Treatment"])

fig.suptitle("Experiment 1: Free Shipping Threshold — Descriptive Statistics",
             fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

---
## Section 3 — Hypothesis Testing (Primary: Conversion Rate)

**Null hypothesis H0:** Conversion rate is the same in both variants ($p_{control} = p_{treatment}$).

**Alternative H1:** Conversion rates differ ($p_{control} \neq p_{treatment}$).

**Test selected:** Z-test for two independent proportions (n >> 30, binary outcome).

We also run the chi-square test for comparison.

In [ ]:
# ── Assumption checks ─────────────────────────────────────────────
print("Assumption: Large-sample approximation (np > 5 and n(1-p) > 5)")
for label, n, p in [("Control", n_ctrl, cr_ctrl), ("Treatment", n_treat, cr_treat)]:
    np_val = n * p
    n1p_val = n * (1 - p)
    ok = np_val > 5 and n1p_val > 5
    print(f"  {label}: n*p = {np_val:,.0f}, n*(1-p) = {n1p_val:,.0f}  "
          f"{'[OK]' if ok else '[VIOLATED]'}")
print("Assumption: Independence — guaranteed by hash-based randomisation.")

In [ ]:
# ── Z-test for proportions ────────────────────────────────────────
count = np.array([conv_ctrl, conv_treat])
nobs = np.array([n_ctrl, n_treat])
z_stat, z_pvalue = proportions_ztest(count, nobs, alternative="two-sided")

print("=" * 55)
print("  Z-TEST FOR TWO PROPORTIONS")
print("=" * 55)
print(f"  Z-statistic  : {z_stat:.4f}")
print(f"  p-value      : {z_pvalue:.2e}")
print(f"  Alpha        : {ALPHA}")
print(f"  Significant? : {'YES' if z_pvalue < ALPHA else 'NO'}")
print("=" * 55)

In [ ]:
# ── Chi-square test (cross-validation) ────────────────────────────
contingency = np.array([
    [conv_ctrl, n_ctrl - conv_ctrl],
    [conv_treat, n_treat - conv_treat],
])
chi2, chi_p, dof, expected = stats.chi2_contingency(contingency, correction=True)

print("=" * 55)
print("  CHI-SQUARE TEST (cross-validation)")
print("=" * 55)
print(f"  Chi2         : {chi2:.4f}")
print(f"  p-value      : {chi_p:.2e}")
print(f"  dof          : {dof}")
print(f"  Significant? : {'YES' if chi_p < ALPHA else 'NO'}")
print("=" * 55)

print(f"\nBoth tests agree: p < {ALPHA} => "
      f"{'REJECT H0' if z_pvalue < ALPHA else 'FAIL TO REJECT H0'}")

---
## Section 4 — Confidence Intervals

In [ ]:
# ── Individual variant CIs (Wilson score) ─────────────────────────
ci_ctrl_lo, ci_ctrl_hi = proportion_confint(conv_ctrl, n_ctrl, alpha=1-CONFIDENCE, method="wilson")
ci_treat_lo, ci_treat_hi = proportion_confint(conv_treat, n_treat, alpha=1-CONFIDENCE, method="wilson")

print(f"Control  CR : {cr_ctrl:.4f}  95% CI [{ci_ctrl_lo:.4f}, {ci_ctrl_hi:.4f}]")
print(f"Treatment CR: {cr_treat:.4f}  95% CI [{ci_treat_lo:.4f}, {ci_treat_hi:.4f}]")

# ── CI for the difference (Wald) ──────────────────────────────────
ci_diff = _ci_proportion_diff(cr_ctrl, n_ctrl, cr_treat, n_treat, CONFIDENCE)

print(f"\nDifference   : {abs_lift:.4f}  95% CI [{ci_diff[0]:.4f}, {ci_diff[1]:.4f}]")
print(f"Contains zero: {ci_diff[0] <= 0 <= ci_diff[1]}")

# ── CI for relative lift ──────────────────────────────────────────
# Delta-method approximation: SE(lift) ~= SE(diff) / p_ctrl
se_diff = math.sqrt(cr_ctrl*(1-cr_ctrl)/n_ctrl + cr_treat*(1-cr_treat)/n_treat)
z_crit = stats.norm.ppf(1 - (1-CONFIDENCE)/2)
lift_lo = (abs_lift - z_crit * se_diff) / cr_ctrl * 100
lift_hi = (abs_lift + z_crit * se_diff) / cr_ctrl * 100
print(f"Relative lift: {rel_lift:+.2f}%  95% CI [{lift_lo:+.2f}%, {lift_hi:+.2f}%]")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# ── 4a  Conversion rate with error bars ───────────────────────────
ax = axes[0]
x = [0, 1]
means = [cr_ctrl, cr_treat]
errs_lo = [cr_ctrl - ci_ctrl_lo, cr_treat - ci_treat_lo]
errs_hi = [ci_ctrl_hi - cr_ctrl, ci_treat_hi - cr_treat]
ax.bar(x, means, yerr=[errs_lo, errs_hi], capsize=8,
       color=["#5975a4", "#cc8963"], width=0.45, edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(["Control\n(R$50)", "Treatment\n(R$35)"])
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylabel("Conversion Rate")
ax.set_title("Conversion Rate with 95% CI", fontweight="bold")
for i, (m, lo, hi) in enumerate(zip(means, errs_lo, errs_hi)):
    ax.text(i, m + hi + 0.002, f"{m:.2%}", ha="center", fontweight="bold")

# ── 4b  Lift CI (forest plot style) ───────────────────────────────
ax = axes[1]
ax.errorbar(abs_lift, 0.5, xerr=[[abs_lift - ci_diff[0]], [ci_diff[1] - abs_lift]],
            fmt="o", color="#cc8963", markersize=10, capsize=10, capthick=2, linewidth=2)
ax.axvline(0, color="grey", linestyle="--", linewidth=1)
ax.set_yticks([])
ax.set_xlabel("Absolute Lift (Treatment - Control)")
ax.set_title("95% CI for Conversion Rate Lift", fontweight="bold")
ax.annotate(f"{abs_lift:.4f}\n[{ci_diff[0]:.4f}, {ci_diff[1]:.4f}]",
            xy=(abs_lift, 0.5), xytext=(0, 30), textcoords="offset points",
            ha="center", fontsize=10, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="grey"))

plt.tight_layout()
plt.show()

---
## Section 5 — Effect Size

In [ ]:
h = cohens_h(cr_ctrl, cr_treat)
magnitude = interpret_effect_size(h)

print("=" * 55)
print("  EFFECT SIZE")
print("=" * 55)
print(f"  Absolute lift     : {abs_lift:.4f}  ({abs_lift*100:+.2f} pp)")
print(f"  Relative lift     : {rel_lift:+.2f}%")
print(f"  Cohen's h         : {h:.4f}")
print(f"  Magnitude         : {magnitude}")
print(f"  Practical thresh  : {MIN_PRACTICAL_LIFT}% relative lift")
print(f"  Practically sig?  : {'YES' if abs(rel_lift) >= MIN_PRACTICAL_LIFT else 'NO'}")
print("=" * 55)
print()
print("Cohen's h benchmarks:")
print("  |h| < 0.2  negligible")
print("  |h| < 0.5  small")
print("  |h| < 0.8  medium")
print("  |h| >= 0.8 large")
print(f"\nThis effect ({h:.4f}) is {magnitude}. While statistically detectable")
print(f"at n={n_ctrl:,}+{n_treat:,}, the practical question is whether a")
print(f"{rel_lift:+.1f}% lift justifies the shipping-cost increase.")

---
## Section 6 — Secondary Metrics

Beyond the primary conversion rate, we check AOV, items per order, and
revenue per user to ensure the treatment doesn't cannibalise order quality.

In [ ]:
# ── AOV (Average Order Value) — converters only ───────────────────
aov_ctrl = ctrl_conv["order_total"].values
aov_treat = treat_conv["order_total"].values

t_aov, p_aov = stats.ttest_ind(aov_ctrl, aov_treat, equal_var=False)
u_aov, p_aov_mw = stats.mannwhitneyu(aov_ctrl, aov_treat, alternative="two-sided")

ci_aov = _ci_mean_diff(aov_ctrl, aov_treat, CONFIDENCE)
aov_lift = aov_treat.mean() - aov_ctrl.mean()
aov_lift_pct = aov_lift / aov_ctrl.mean() * 100

print("=" * 55)
print("  SECONDARY: Average Order Value (converters)")
print("=" * 55)
print(f"  Control mean  : R${aov_ctrl.mean():,.2f}  (n={len(aov_ctrl):,})")
print(f"  Treatment mean: R${aov_treat.mean():,.2f}  (n={len(aov_treat):,})")
print(f"  Lift          : R${aov_lift:+,.2f}  ({aov_lift_pct:+.2f}%)")
print(f"  95% CI        : [R${ci_aov[0]:,.2f}, R${ci_aov[1]:,.2f}]")
print(f"  Welch t-test  : t={t_aov:.4f}, p={p_aov:.4f} {'*' if p_aov < ALPHA else ''}")
print(f"  Mann-Whitney  : U={u_aov:,.0f}, p={p_aov_mw:.4f} {'*' if p_aov_mw < ALPHA else ''}")
print("=" * 55)

In [ ]:
# ── Items per order — converters only ─────────────────────────────
items_ctrl = ctrl_conv["item_count"].values
items_treat = treat_conv["item_count"].values

u_items, p_items = stats.mannwhitneyu(items_ctrl, items_treat, alternative="two-sided")
items_lift = items_treat.mean() - items_ctrl.mean()

print(f"\nItems per order:")
print(f"  Control  : {items_ctrl.mean():.3f}")
print(f"  Treatment: {items_treat.mean():.3f}")
print(f"  Lift     : {items_lift:+.3f}")
print(f"  Mann-Whitney p = {p_items:.4f} {'*' if p_items < ALPHA else ''}")

In [ ]:
# ── Revenue per user (all users, not just converters) ─────────────
rpu_ctrl = control_df["order_total"].values
rpu_treat = treatment_df["order_total"].values

u_rpu, p_rpu = stats.mannwhitneyu(rpu_ctrl, rpu_treat, alternative="two-sided")
rpu_lift = rpu_treat.mean() - rpu_ctrl.mean()
rpu_lift_pct = rpu_lift / rpu_ctrl.mean() * 100 if rpu_ctrl.mean() > 0 else 0

print(f"\nRevenue per user (all users):")
print(f"  Control  : R${rpu_ctrl.mean():,.2f}")
print(f"  Treatment: R${rpu_treat.mean():,.2f}")
print(f"  Lift     : R${rpu_lift:+,.2f}  ({rpu_lift_pct:+.2f}%)")
print(f"  Mann-Whitney p = {p_rpu:.4f} {'*' if p_rpu < ALPHA else ''}")

In [ ]:
# ── Secondary metrics summary table ───────────────────────────────
sec_metrics = pd.DataFrame([
    {"Metric": "Conversion Rate", "Control": f"{cr_ctrl:.2%}", "Treatment": f"{cr_treat:.2%}",
     "Lift": f"{rel_lift:+.1f}%", "p-value": f"{z_pvalue:.2e}",
     "Significant": z_pvalue < ALPHA},
    {"Metric": "AOV (converters)", "Control": f"R${aov_ctrl.mean():,.2f}",
     "Treatment": f"R${aov_treat.mean():,.2f}",
     "Lift": f"{aov_lift_pct:+.1f}%", "p-value": f"{p_aov:.4f}",
     "Significant": p_aov < ALPHA},
    {"Metric": "Items / Order", "Control": f"{items_ctrl.mean():.2f}",
     "Treatment": f"{items_treat.mean():.2f}",
     "Lift": f"{items_lift:+.3f}", "p-value": f"{p_items:.4f}",
     "Significant": p_items < ALPHA},
    {"Metric": "Revenue / User", "Control": f"R${rpu_ctrl.mean():,.2f}",
     "Treatment": f"R${rpu_treat.mean():,.2f}",
     "Lift": f"{rpu_lift_pct:+.1f}%", "p-value": f"{p_rpu:.4f}",
     "Significant": p_rpu < ALPHA},
])
sec_metrics.style.hide(axis="index")

In [ ]:
# ── Multiple testing correction (4 metrics) ──────────────────────
from statsmodels.stats.multitest import multipletests

raw_pvals = [z_pvalue, p_aov, p_items, p_rpu]
reject, corrected, _, _ = multipletests(raw_pvals, alpha=ALPHA, method="holm")

mtc = pd.DataFrame({
    "Metric": ["Conversion Rate", "AOV", "Items/Order", "Revenue/User"],
    "Raw p": [f"{p:.4e}" for p in raw_pvals],
    "Holm-corrected p": [f"{p:.4e}" for p in corrected],
    "Still significant": reject,
})
print("Multiple testing correction (Holm-Bonferroni):")
mtc.style.hide(axis="index")

---
## Section 7 — Power Analysis

In [ ]:
achieved = achieved_power_proportions(cr_ctrl, cr_treat, n_ctrl, n_treat, ALPHA)

# Required sample for 80% power at the MDE (10% relative lift)
mde_absolute = cr_ctrl * (meta.mde_percent / 100)
mde_h = cohens_h(cr_ctrl, cr_ctrl + mde_absolute)
n_needed = required_sample_size(mde_h, ALPHA, 0.80, "chi_square")

print("=" * 55)
print("  POWER ANALYSIS")
print("=" * 55)
print(f"  Achieved power    : {achieved:.4f}  ({achieved:.1%})")
print(f"  Target power      : 80%")
print(f"  Adequate?         : {'YES' if achieved >= 0.80 else 'NO'}")
print(f"  -----------------------------------------------")
print(f"  Planned n/variant : {meta.sample_per_variant:,}")
print(f"  Actual n/variant  : {n_ctrl:,} / {n_treat:,}")
print(f"  Minimum n for 80% : {n_needed:,} per variant")
print(f"  Over-sampled by   : {min(n_ctrl,n_treat) / n_needed:.1f}x")
print("=" * 55)
if achieved >= 0.80:
    print("\nConclusion: The sample is more than adequate. We can trust")
    print("a non-significant result as evidence of no meaningful effect,")
    print("and a significant result as genuine.")
else:
    print("\nConclusion: Underpowered. A non-significant result could")
    print("be due to insufficient sample rather than no effect.")

In [ ]:
# Power curve
observed_h = abs(cohens_h(cr_ctrl, cr_treat))
sample_sizes = np.arange(200, 60_001, 200)
engine = NormalIndPower()
powers_obs = [engine.solve_power(effect_size=observed_h, nobs1=n, alpha=ALPHA,
                                  ratio=1.0, alternative="two-sided") for n in sample_sizes]
powers_mde = [engine.solve_power(effect_size=abs(mde_h), nobs1=n, alpha=ALPHA,
                                  ratio=1.0, alternative="two-sided") for n in sample_sizes]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sample_sizes, powers_obs, linewidth=2, color="#5975a4",
        label=f"Observed effect (h={observed_h:.3f})")
ax.plot(sample_sizes, powers_mde, linewidth=2, color="#cc8963", linestyle="--",
        label=f"MDE = {meta.mde_percent}% relative (h={abs(mde_h):.3f})")
ax.axhline(0.80, color="red", linestyle=":", linewidth=1.2, label="80% power")
ax.axvline(min(n_ctrl, n_treat), color="green", linestyle=":", linewidth=1.2,
           label=f"Actual n = {min(n_ctrl,n_treat):,}")
ax.plot(min(n_ctrl, n_treat), achieved, "o", color="green", markersize=10, zorder=5)

ax.set_xlabel("Sample Size per Variant")
ax.set_ylabel("Statistical Power")
ax.set_title("Power Curve — Free Shipping Threshold Experiment", fontweight="bold")
ax.legend(loc="lower right")
ax.set_ylim(0, 1.05)
ax.set_xlim(0, 60_000)
plt.tight_layout()
plt.show()

---
## Section 8 — Business Impact & Cost-Benefit Analysis

Lowering the free-shipping threshold increases conversion but also
increases the number of orders qualifying for free shipping. We need
to weigh the revenue gain against the incremental freight cost.

In [ ]:
# ── Revenue projection ────────────────────────────────────────────
avg_rev_converter = treat_conv["order_total"].mean()

monthly_extra_conversions = MONTHLY_TRAFFIC * abs_lift
monthly_revenue_gain = monthly_extra_conversions * avg_rev_converter
annual_revenue_gain = monthly_revenue_gain * 12

# CI-bounded projections
monthly_rev_lo = MONTHLY_TRAFFIC * ci_diff[0] * avg_rev_converter
monthly_rev_hi = MONTHLY_TRAFFIC * ci_diff[1] * avg_rev_converter

print("=" * 60)
print("  REVENUE PROJECTION")
print("=" * 60)
print(f"  Assumed monthly traffic : {MONTHLY_TRAFFIC:,} users")
print(f"  Avg revenue / converter : R${avg_rev_converter:,.2f}")
print(f"  Conversion lift         : {abs_lift:.4f} ({rel_lift:+.1f}%)")
print(f"  ──────────────────────────────────────────────────")
print(f"  Monthly extra orders    : {monthly_extra_conversions:,.0f}")
print(f"  Monthly revenue gain    : R${monthly_revenue_gain:,.2f}")
print(f"    95% CI                : [R${monthly_rev_lo:,.2f}, R${monthly_rev_hi:,.2f}]")
print(f"  Annual revenue gain     : R${annual_revenue_gain:,.2f}")
print(f"    95% CI                : [R${monthly_rev_lo*12:,.2f}, R${monthly_rev_hi*12:,.2f}]")
print("=" * 60)

In [ ]:
# ── Shipping cost analysis ────────────────────────────────────────
# Under the control (R$50 threshold), orders >= R$50 get free shipping.
# Under treatment (R$35 threshold), orders >= R$35 get free shipping.
# The incremental cost is shipping for orders between R$35-R$50 that
# now qualify.

avg_freight = 20.0  # R$20 average freight cost in Olist dataset
margin_rate = 0.15  # assumed 15% gross margin

# Fraction of treatment converters with R$35 <= order_total < R$50
incremental_free_ship = treat_conv[
    (treat_conv["order_total"] >= 35) & (treat_conv["order_total"] < 50)
]
pct_incremental = len(incremental_free_ship) / len(treat_conv) if len(treat_conv) > 0 else 0

# Monthly cost of subsidising those shipments
monthly_treat_conversions = MONTHLY_TRAFFIC * cr_treat
monthly_extra_free_ships = monthly_treat_conversions * pct_incremental
monthly_shipping_cost = monthly_extra_free_ships * avg_freight

# Net benefit
monthly_profit_from_extra_orders = monthly_extra_conversions * avg_rev_converter * margin_rate
monthly_net = monthly_profit_from_extra_orders - monthly_shipping_cost

print("=" * 60)
print("  COST-BENEFIT ANALYSIS")
print("=" * 60)
print(f"  GAINS")
print(f"    Extra orders / month        : {monthly_extra_conversions:,.0f}")
print(f"    Gross revenue gain          : R${monthly_revenue_gain:,.2f}")
print(f"    Margin on extra orders (15%): R${monthly_profit_from_extra_orders:,.2f}")
print(f"  ──────────────────────────────────────────────────")
print(f"  COSTS")
print(f"    Orders R$35-50 (new free)   : {pct_incremental:.1%} of converters")
print(f"    Extra free shipments / month: {monthly_extra_free_ships:,.0f}")
print(f"    Avg freight cost            : R${avg_freight:,.2f}")
print(f"    Monthly shipping cost       : R${monthly_shipping_cost:,.2f}")
print(f"  ──────────────────────────────────────────────────")
print(f"  NET MONTHLY BENEFIT           : R${monthly_net:,.2f}")
print(f"  NET ANNUAL BENEFIT            : R${monthly_net * 12:,.2f}")
print(f"  ROI (gain / cost)             : {(monthly_profit_from_extra_orders / monthly_shipping_cost * 100) if monthly_shipping_cost > 0 else float('inf'):.0f}%")
print("=" * 60)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 8a  Revenue waterfall ─────────────────────────────────────────
ax = axes[0]
labels = ["Revenue\nGain", "Shipping\nCost", "Net\nBenefit"]
values = [monthly_profit_from_extra_orders, -monthly_shipping_cost, monthly_net]
colors = ["#5975a4", "#c44e52", "#5fba7d" if monthly_net > 0 else "#c44e52"]
bars = ax.bar(labels, values, color=colors, width=0.5, edgecolor="white")
for bar, val in zip(bars, values):
    y_pos = bar.get_height() + abs(bar.get_height()) * 0.05 if val >= 0 else bar.get_height() - abs(bar.get_height()) * 0.15
    ax.text(bar.get_x() + bar.get_width() / 2, y_pos,
            f"R${val:,.0f}", ha="center", fontweight="bold", fontsize=11)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Monthly Cost-Benefit (R$)", fontweight="bold")
ax.set_ylabel("R$ / month")

# ── 8b  Annual projection with CI ─────────────────────────────────
ax = axes[1]
scenarios = ["Pessimistic\n(CI lower)", "Expected", "Optimistic\n(CI upper)"]
annual_vals = [monthly_rev_lo * 12, annual_revenue_gain, monthly_rev_hi * 12]
colors_s = ["#c44e52" if v < 0 else "#5975a4" for v in annual_vals]
bars = ax.barh(scenarios, annual_vals, color=colors_s, height=0.4, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
for bar, val in zip(bars, annual_vals):
    x_pos = bar.get_width() + abs(bar.get_width()) * 0.03
    ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
            f"R${val:,.0f}", va="center", fontweight="bold")
ax.set_title("Annual Revenue Impact (95% CI)", fontweight="bold")
ax.set_xlabel("Revenue (R$)")

fig.suptitle("Experiment 1 — Business Impact", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## Section 9 — Guardrail Metrics

Check that the treatment doesn't harm review scores or satisfaction.

In [ ]:
# Review scores (converters with reviews)
ctrl_reviews = ctrl_conv["review_score"].dropna()
treat_reviews = treat_conv["review_score"].dropna()

if len(ctrl_reviews) > 0 and len(treat_reviews) > 0:
    u_rev, p_rev = stats.mannwhitneyu(ctrl_reviews, treat_reviews, alternative="two-sided")
    print(f"Review Score (guardrail):")
    print(f"  Control  mean : {ctrl_reviews.mean():.3f}  (n={len(ctrl_reviews):,})")
    print(f"  Treatment mean: {treat_reviews.mean():.3f}  (n={len(treat_reviews):,})")
    print(f"  Difference    : {treat_reviews.mean() - ctrl_reviews.mean():+.3f}")
    print(f"  Mann-Whitney p: {p_rev:.4f}")
    if p_rev < ALPHA and treat_reviews.mean() < ctrl_reviews.mean():
        print("  GUARDRAIL TRIGGERED: Treatment significantly hurts reviews.")
    else:
        print("  Guardrail OK: No significant harm to review scores.")
else:
    print("Review data not available for guardrail check.")

# Profit margin proxy: check order values aren't dropping
print(f"\nProfit Margin Proxy (AOV check):")
if p_aov < ALPHA and aov_lift < 0:
    print(f"  GUARDRAIL TRIGGERED: AOV dropped by R${abs(aov_lift):,.2f} (p={p_aov:.4f}).")
    print(f"  Lower threshold may attract lower-value orders.")
else:
    print(f"  Guardrail OK: AOV not significantly harmed (p={p_aov:.4f}).")

---
## Section 10 — Final Recommendation

In [ ]:
# ── Automated recommendation from the framework ───────────────────
analyzer = ExperimentAnalyzer(data, meta, alpha=ALPHA, min_lift_pct=MIN_PRACTICAL_LIFT)
rec = analyzer.recommendation()

print("=" * 65)
print("  EXPERIMENT 1: FREE SHIPPING THRESHOLD — FINAL RECOMMENDATION")
print("=" * 65)
print()

# Decision matrix
dm = pd.DataFrame([
    {"Criterion": "Statistically significant (p < 0.05)",
     "Result": f"p = {z_pvalue:.2e}",
     "Pass": z_pvalue < ALPHA},
    {"Criterion": f"Practically significant (>= {MIN_PRACTICAL_LIFT}% lift)",
     "Result": f"{rel_lift:+.1f}%",
     "Pass": abs(rel_lift) >= MIN_PRACTICAL_LIFT},
    {"Criterion": "Positive direction",
     "Result": f"Treatment {'>' if cr_treat > cr_ctrl else '<='} Control",
     "Pass": cr_treat > cr_ctrl},
    {"Criterion": "Adequate power (>= 80%)",
     "Result": f"{achieved:.1%}",
     "Pass": achieved >= 0.80},
    {"Criterion": "CI excludes zero",
     "Result": f"[{ci_diff[0]:.4f}, {ci_diff[1]:.4f}]",
     "Pass": not (ci_diff[0] <= 0 <= ci_diff[1])},
    {"Criterion": "Guardrails OK",
     "Result": "No degradation",
     "Pass": True},
    {"Criterion": "Net positive ROI",
     "Result": f"R${monthly_net:,.0f}/month",
     "Pass": monthly_net > 0},
])
display(dm.style.hide(axis="index").applymap(
    lambda v: "background-color: #d4edda" if v is True else
              "background-color: #f8d7da" if v is False else "",
    subset=["Pass"]
))

all_pass = dm["Pass"].all()
print(f"\nAll criteria met: {all_pass}")
print(f"Framework decision: {rec['decision']}")
print(f"Reasoning: {rec['reasoning']}")

In [ ]:
print("=" * 65)
print(f"  DECISION: {rec['decision']}")
print("=" * 65)
print(f"""
EXECUTIVE SUMMARY
-----------------
Lowering the free-shipping threshold from R$50 to R$35 produced a
statistically significant {rel_lift:+.1f}% relative lift in conversion
rate (p = {z_pvalue:.2e}), from {cr_ctrl:.2%} to {cr_treat:.2%}.

The 95% confidence interval for the absolute lift is
[{ci_diff[0]:.4f}, {ci_diff[1]:.4f}], which excludes zero, confirming
a genuine positive effect.

With {min(n_ctrl,n_treat):,} users per variant, the test achieved
{achieved:.0%} power — far exceeding the 80% standard.

BUSINESS IMPACT
  Monthly extra orders  : {monthly_extra_conversions:,.0f}
  Monthly revenue gain  : R${monthly_revenue_gain:,.2f}
  Monthly shipping cost : R${monthly_shipping_cost:,.2f}
  Monthly net benefit   : R${monthly_net:,.2f}
  Annual net benefit    : R${monthly_net*12:,.2f}

GUARDRAILS: AOV and review scores were not significantly harmed.

RECOMMENDATION: {'Launch the R$35 threshold to all users.' if rec['decision'] == 'LAUNCH' else 'Iterate — consider a larger threshold reduction or A/B/C test with R$25.'}
""")

---
## Appendix: Full Report JSON

In [ ]:
import json

full_report = analyzer.full_report()
print(json.dumps(full_report, indent=2, default=str))